# 07 · Pipelines with Ray

*The phase 1 ingestion rebuilt as a pipeline that can run on one laptop or a cluster.*

Phase 1 built the station data with a thread pool and a few conventions: one unit of work per station-year, raw files never touched, outputs written atomically. That was enough for 165 stations. It isn't a pipeline yet. Nothing records what was done or with which code, a backfill and next month's update are different scripts, and the only way to scale is a bigger laptop.

This notebook rebuilds it with **Ray**. Ray is a Python framework for running functions and stateful objects across many cores or machines. It was built at Berkeley's RISELab for workloads that don't fit neatly into SQL-style batch jobs, like reinforcement learning, hyperparameter search and model serving, and it has become a common backbone for ML data pipelines.

The goal is to learn Ray's model by using it on a real job. Three ideas carry most of it:

- A **task** is a function run somewhere in the cluster. You call it and get back a reference to its result, not the result.
- An **object reference** is a handle to a value in Ray's shared **object store**. Passing references between tasks lets Ray move data only when and where it's needed.
- An **actor** is a stateful worker: a Python object living in its own process whose methods run as tasks. It's how you keep something expensive, like an open connection, alive between calls.

On top of those sits **Ray Data**, a library for streaming datasets through a chain of transformations, which we use at the end.

**Read first:** Ray's [Key concepts](https://docs.ray.io/en/latest/ray-core/key-concepts.html), [Tasks](https://docs.ray.io/en/latest/ray-core/tasks.html), [Actors](https://docs.ray.io/en/latest/ray-core/actors.html), [Fault tolerance](https://docs.ray.io/en/latest/ray-core/fault-tolerance.html) and [Ray Data key concepts](https://docs.ray.io/en/latest/data/key-concepts.html). About 45 minutes.

In [ ]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import pandas as pd
from pathlib import Path
import ray
from wxlab import stations as st, pipeline as pl

RAW = Path(DATA_DIR) / "raw" / "isd"
OUT = Path(DATA_DIR) / "work" / "ray"                 # a separate output tree, so phase 1's data is untouched
STD, MANIFEST = OUT / "standardized", OUT / "manifest.parquet"
station_ids = sorted(p.stem for p in (Path(DATA_DIR) / "standardized" / "isd" / "year=2023").glob("*.parquet"))
ray.init(num_cpus=8, ignore_reinit_error=True, include_dashboard=False)
print(len(station_ids), "stations;", ray.available_resources().get("CPU"), "CPUs for Ray")

## Step 1 · Units, a manifest and a plan

The framework-neutral parts live in `wxlab.pipeline`. A **unit** is one station-year. The **manifest** is a table with a row per attempted unit: its key, a fingerprint of its input file, the code version, the outcome and the output path. **Planning** compares the units we want with the manifest and returns only what's missing, failed, or out of date because its input or the code changed.

That last part is what makes reprocessing safe. Bump `CODE_VERSION` and every unit is planned again. Re-download one raw file and only its unit is.

In [ ]:
wanted = pl.units_for(station_ids, range(2021, 2025))
manifest = pl.load_manifest(MANIFEST)
todo = pl.plan(wanted, manifest, RAW)
print(f"{len(wanted)} units wanted, {len(manifest)} manifest rows, {len(todo)} to run")

## Step 2 · The backfill as Ray tasks

`pl.run_unit` is a plain function with no Ray in it. `ray.remote` turns it into a task. Calling `.remote()` returns an object reference immediately and Ray schedules the work on a free CPU. `ray.wait` hands back results as they finish rather than in submission order, so the manifest can be written in batches as work completes. If the notebook dies halfway, the next run's plan picks up exactly what's left.

`max_retries` tells Ray to rerun a task if its worker process dies, which is the cheap kind of fault tolerance. It's safe only because the unit is idempotent: running it twice gives the same file.

In [ ]:
run_unit = ray.remote(max_retries=2)(pl.run_unit)

t0 = time.time()
pending = [run_unit.remote(u, RAW, STD) for u in todo]
rows = []
while pending:
    done, pending = ray.wait(pending, num_returns=min(32, len(pending)), timeout=30)
    rows += ray.get(done)
    if len(rows) % 128 < 32:
        manifest = pl.record(MANIFEST, rows); rows = []
manifest = pl.record(MANIFEST, rows)
print(f"{len(todo)} units in {time.time() - t0:.0f} s")
manifest.groupby("status").agg(units=("key", "size"), rows=("rows", "sum"), seconds=("seconds", "sum"))

*Draft note: report wall time against summed task seconds (the parallel speed-up) and compare with phase 1's thread pool. Parsing is CPU-bound pandas work, so threads were limited by Python's global interpreter lock and processes should do better. Run the cell a second time to show that the plan is empty.*

## Step 3 · Incremental runs use the same code

A live pipeline gets one new piece at a time. To simulate it without touching the raw files (they stay immutable), we make the manifest forget 2024, as if that year had just arrived. The plan finds exactly those units and the same task runs them. No second script.

In [ ]:
m = pl.load_manifest(MANIFEST)
m[m.year != 2024].to_parquet(MANIFEST, index=False)   # pretend 2024 is new
todo = pl.plan(wanted, pl.load_manifest(MANIFEST), RAW)
print(len(todo), "units planned; years:", sorted({u.year for u in todo}))
manifest = pl.record(MANIFEST, ray.get([run_unit.remote(u, RAW, STD) for u in todo]))

For a real live feed the trigger would be an event rather than a loop. NOAA publishes a notification for every new object in its buckets, so a small listener can turn "new file landed" into "plan this unit". The unit, the plan and the manifest don't change.

## Step 4 · Actors for state that should live between calls

Phase 1 learned that opening a new HTTPS connection for every request failed at scale, and one reused connection per worker fixed it. In Ray that's an **actor**: a worker process that holds state. A small pool of downloader actors, each with its own session, does what the thread-local sessions did, and it works the same way on a cluster.

In [ ]:
@ray.remote
class Downloader:
    def __init__(self):
        import requests
        self.session = requests.Session()               # opened once per actor and reused

    def head(self, station_id, year):
        url = f"https://{st.BUCKET}.s3.amazonaws.com/{year}/{station_id}.csv"
        r = self.session.head(url, timeout=30)
        return station_id, year, r.status_code, int(r.headers.get("Content-Length", 0))

pool = ray.util.ActorPool([Downloader.remote() for _ in range(4)])
sample = [(s, 2024) for s in station_ids[:40]]
sizes = list(pool.map_unordered(lambda a, v: a.head.remote(*v), sample))
pd.DataFrame(sizes, columns=["station_id", "year", "status", "bytes"]).describe()

Only `HEAD` requests here, so nothing is downloaded again. The shape is the point: the session is opened once per actor and lives as long as the actor.

## Step 5 · Ray Data for the next stage

Tasks are good for "run this function on each unit". The QA stage is different. It reads many units and groups them differently, by month across all stations, because the neighbour checks need every station's data for the same hours. **Ray Data** handles that kind of whole-dataset step. It reads files as a stream of blocks, applies transformations batch by batch, and uses backpressure so a slow stage doesn't let the fast ones fill up memory.

As a small example, compute each station's monthly completeness and mean temperature straight from the standardized files.

In [ ]:
ds = ray.data.read_parquet(str(STD), columns=["station_id", "time", "air_temperature"])

def monthly(batch: pd.DataFrame) -> pd.DataFrame:
    batch["month"] = batch.time.dt.strftime("%Y-%m")
    return batch.groupby(["station_id", "month"]).agg(hours=("air_temperature", "count"),
                                                      mean_t=("air_temperature", "mean")).reset_index()

summary = ds.map_batches(monthly, batch_format="pandas").to_pandas()
summary = summary.groupby(["station_id", "month"]).agg(hours=("hours", "sum")).reset_index()  # blocks can split a month
summary.describe()

Notice the second `groupby`. A block can hold part of a station-month, so per-block results have to be combined again. That's a small taste of the **shuffle** problem: any step that groups by a key the data isn't already partitioned by needs data moved between workers. Ray Data has `groupby` for this, but large shuffles are where Spark's engine is more mature. Notebook 08 shows the same step there.

## Where Ray fits

These are the documented strengths and limits, with what this job showed in brackets.

- **Good at:** plain Python functions and objects scaled out with little ceremony [the unit function needed no changes]. Mixed workloads where tasks differ in size, use GPUs or hold state [actors for connections]. ML stages in the same system, like Ray Train for training and Ray Serve for serving.
- **Less natural for:** SQL-style joins and very large shuffles, where Spark's optimizer and shuffle service are the mature choice. Event-time streaming with windows and watermarks, which Ray doesn't model, while Beam and Flink do (notebook 08).
- **Operational facts:** a Ray cluster is long-lived processes with a head node. Work is lost if the head node fails unless you configure for it. Lineage-based retries only cover tasks, not actor state.

*Draft note: add a one-line timing comparison with phase 1 once run.*

## What surprised me

*Draft: fill in after running.*

## Check yourself

<details><summary>Why does the plan fingerprint the input and record the code version?</summary>
So "done" means done with this input and this code. If either changes, the unit is planned again. Without them a pipeline either redoes everything or silently keeps stale output.
</details>

<details><summary>Why is it safe to give the task max_retries=2?</summary>
Because the unit is idempotent and writes atomically. A retry produces the same file and a killed attempt leaves no half-written output.
</details>

<details><summary>Why is QA a different shape of work from standardizing?</summary>
Standardizing needs one station-year. Neighbour checks need every station for the same hours. The data must be regrouped from by-station to by-time, which is a shuffle.
</details>

New terms from this notebook are in [`docs/glossary.md`](../docs/glossary.md).

In [ ]:
ray.shutdown()